In [3]:

%pip install openpyxl



  Using cached openpyxl-3.1.5-py2.py3-none-any.whl (250 kB)
  Using cached et_xmlfile-2.0.0-py3-none-any.whl (18 kB)
Note: you may need to restart the kernel to use updated packages.



[notice] A new release of pip is available: 23.1.2 -> 26.2.1
[notice] To update, run: python.exe -m pip install --upgrade pip


In [4]:
app = pd.read_excel(
    "AssessmentDataScientist.xlsx",
    sheet_name="Application Data"
)

performance = pd.read_excel(
    "AssessmentDataScientist.xlsx",
    sheet_name="Loan Performance"
)

print("Application Data Shape:", app.shape)
print("Loan Performance Shape:", performance.shape)

Application Data Shape: (647, 31)
Loan Performance Shape: (1285, 2)


c:\Users\Admin\AppData\Local\Programs\Python\Python311\Lib\site-packages\openpyxl\worksheet\_reader.py:329: UserWarning: Unknown extension is not supported and will be removed
  warn(msg)
c:\Users\Admin\AppData\Local\Programs\Python\Python311\Lib\site-packages\openpyxl\worksheet\_reader.py:329: UserWarning: Unknown extension is not supported and will be removed
  warn(msg)


In [5]:
print("APPLICATION DATA")
display(app.head())

print("LOAN PERFORMANCE")
display(performance.head())

APPLICATION DATA


,customer_id,amount_requested,birth_date,status,residence_rent_or_own,monthly_rent_amount,bank_account_direct_deposit,application_when,loan_duration,payment_ach,...,payment_frequency,home_phone_type,other_phone_type,how_use_money,monthly_income_amount,raw_l2c_score,raw_FICO_telecom,raw_FICO_retail,raw_FICO_bank_card,raw_FICO_money
0,9ece67d6c5,500,1991-08-21,Approved,True,0,True,2010-10-16 14:06:05,3,True,...,Semi-monthly,Mobile,NaN,Other,1560,614,574,600,656,561
1,5c2c402094,500,1990-07-24,Approved,True,0,False,2010-10-17 13:01:53,6,True,...,Bi-weekly,Mobile,NaN,Other,900,708,501,550,651,563
2,e6254cad30,400,1980-09-13,Approved,True,620,True,2010-10-17 19:52:34,6,True,...,Bi-weekly,Mobile,NaN,Other,1434,687,522,561,661,598
3,49fb42f51d,500,1976-09-10,Approved,False,785,False,2010-10-18 07:05:00,4,True,...,Bi-weekly,Mobile,NaN,Other,1600,616,560,552,634,591
4,195fbe5739,300,1979-12-01,Approved,True,700,True,2010-10-19 10:58:40,4,True,...,Bi-weekly,Mobile,Home,Other,1360,681,603,654,659,636


LOAN PERFORMANCE


,idLoan,flgGood
0,0DBA6B011D-01,Bad
1,96ACBBA7ED-01,Bad
2,289421D82A-01,Bad
3,9ECE67D6C5-01,Good
4,5C2C402094-01,Bad


In [6]:
print("APPLICATION COLUMNS AND TYPES")
display(app.dtypes.to_frame("Data Type"))

print("LOAN PERFORMANCE COLUMNS AND TYPES")
display(performance.dtypes.to_frame("Data Type"))

APPLICATION COLUMNS AND TYPES


,Data Type
customer_id,object
amount_requested,int64
birth_date,datetime64[ns]
status,object
residence_rent_or_own,bool
monthly_rent_amount,int64
bank_account_direct_deposit,bool
application_when,datetime64[ns]
loan_duration,int64
payment_ach,bool


LOAN PERFORMANCE COLUMNS AND TYPES


,Data Type
idLoan,object
flgGood,object


In [7]:
print("Missing values in Application Data:")
missing_app = app.isnull().sum().sort_values(ascending=False)
display(missing_app[missing_app > 0].to_frame("Missing Count"))

print("Exact duplicate application rows:", app.duplicated().sum())

print("Loan outcome counts:")
display(performance["flgGood"].value_counts(dropna=False).to_frame("Count"))

print("Duplicate loan IDs:", performance["idLoan"].duplicated().sum())


Missing values in Application Data:


,Missing Count
other_phone_type,285
payment_amount_approved,20
how_use_money,2
bank_account_duration,1


Exact duplicate application rows: 0
Loan outcome counts:


,Count
flgGood,
Good,769
Bad,516


Duplicate loan IDs: 0


In [8]:

# Inspect identifiers in both datasets
print("Application customer IDs:")
display(app[["customer_id"]].head(10))

print("Loan IDs:")
display(performance[["idLoan", "flgGood"]].head(10))

# Extract the customer-like portion of each loan ID
performance["customer_id_check"] = (
    performance["idLoan"]
    .astype(str)
    .str.replace(r"-\d+$", "", regex=True)
    .str.strip()
    .str.lower()
)

app_ids = app["customer_id"].astype(str).str.strip().str.lower()

print("\nApplication ID examples:")
print(app_ids.head(10).tolist())

print("\nExtracted loan customer ID examples:")
print(performance["customer_id_check"].head(10).tolist())

# Check the overlap
app_id_set = set(app_ids)
loan_id_set = set(performance["customer_id_check"])

print("\nUnique application IDs:", len(app_id_set))
print("Unique extracted loan customer IDs:", len(loan_id_set))
print("Matching IDs:", len(app_id_set & loan_id_set))
print("Application IDs without loan matches:", len(app_id_set - loan_id_set))
print("Loan customer IDs without application matches:", len(loan_id_set - app_id_set))

# Check customers with multiple historical loans
loan_counts = performance.groupby("customer_id_check").size()

print("\nCustomers with multiple loans:")
display(loan_counts[loan_counts > 1].sort_values(ascending=False).head(10))


Application customer IDs:


,customer_id
0,9ece67d6c5
1,5c2c402094
2,e6254cad30
3,49fb42f51d
4,195fbe5739
5,b5286ff5b4
6,d06841c8cf
7,90e4cdc381
8,c1a1065b7a
9,0dba6b011d


Loan IDs:


,idLoan,flgGood
0,0DBA6B011D-01,Bad
1,96ACBBA7ED-01,Bad
2,289421D82A-01,Bad
3,9ECE67D6C5-01,Good
4,5C2C402094-01,Bad
5,E6254CAD30-01,Good
6,AE03151FEE-01,Good
7,E55A9A7293-01,Bad
8,CFE9E0DC5E-01,Good
9,3744B9845D-01,Good



Application ID examples:
['9ece67d6c5', '5c2c402094', 'e6254cad30', '49fb42f51d', '195fbe5739', 'b5286ff5b4', 'd06841c8cf', '90e4cdc381', 'c1a1065b7a', '0dba6b011d']

Extracted loan customer ID examples:
['0dba6b011d', '96acbba7ed', '289421d82a', '9ece67d6c5', '5c2c402094', 'e6254cad30', 'ae03151fee', 'e55a9a7293', 'cfe9e0dc5e', '3744b9845d']

Unique application IDs: 647
Unique extracted loan customer IDs: 1269
Matching IDs: 640
Application IDs without loan matches: 7
Loan customer IDs without application matches: 629

Customers with multiple loans:


customer_id_check
0f9800a7ff    2
2ac47a61e0    2
34ddb612ce    2
418abe2912    2
4dad1a9de6    2
70409dc892    2
a15e74274b    2
ae03151fee    2
b1a2837477    2
b3f9914dbb    2
dtype: int64

In [9]:

# Investigate the relationship between application IDs and loan IDs
app_ids = (
    app["customer_id"]
    .astype(str)
    .str.strip()
    .str.lower()
)

loan_customer_ids = (
    performance["idLoan"]
    .astype(str)
    .str.replace(r"-\d+$", "", regex=True)
    .str.strip()
    .str.lower()
)

print("Unique application IDs:", app_ids.nunique())
print("Unique extracted loan customer IDs:", loan_customer_ids.nunique())

print("Matching IDs:",
      len(set(app_ids) & set(loan_customer_ids)))

print("Application IDs without matches:",
      len(set(app_ids) - set(loan_customer_ids)))

print("Loan IDs without matching applications:",
      len(set(loan_customer_ids) - set(app_ids)))

# Check for multiple historical loans per extracted customer
loan_counts = loan_customer_ids.value_counts()

print("\nCustomers with multiple loans:")
display(loan_counts[loan_counts > 1].head(10))


Unique application IDs: 647
Unique extracted loan customer IDs: 1269
Matching IDs: 640
Application IDs without matches: 7
Loan IDs without matching applications: 629

Customers with multiple loans:


idLoan
0f9800a7ff    2
4dad1a9de6    2
b1a2837477    2
d06841c8cf    2
b3f9914dbb    2
b86063bf54    2
418abe2912    2
a15e74274b    2
dfb2f1ec63    2
d7001aa34e    2
Name: count, dtype: int64

In [10]:

# Keep the original datasets unchanged
app_ids = app["customer_id"].astype(str).str.strip().str.lower()

loan_check = performance[["idLoan", "flgGood"]].copy()
loan_check["customer_id_check"] = (
    loan_check["idLoan"]
    .astype(str)
    .str.replace(r"-\d+$", "", regex=True)
    .str.strip()
    .str.lower()
)

# 1. Inspect matching examples
matching_ids = set(app_ids) & set(loan_check["customer_id_check"])

print("Matched ID examples:")
display(
    loan_check[
        loan_check["customer_id_check"].isin(matching_ids)
    ].head(10)
)

# 2. Inspect application IDs without loan outcomes
unmatched_app = app.loc[~app_ids.isin(matching_ids)]

print("Unmatched application records:", len(unmatched_app))
display(unmatched_app[["customer_id"]].head(10))

# 3. Inspect customers whose loans have different outcomes
outcome_counts = (
    loan_check.groupby("customer_id_check")["flgGood"]
    .nunique()
)

conflicting_ids = outcome_counts[outcome_counts > 1].index

print("Customers with conflicting loan outcomes:", len(conflicting_ids))
display(
    loan_check[
        loan_check["customer_id_check"].isin(conflicting_ids)
    ].sort_values("customer_id_check").head(20)
)

# 4. Inspect all application column names
print("Application columns:")
print(app.columns.tolist())


Matched ID examples:


,idLoan,flgGood,customer_id_check
0,0DBA6B011D-01,Bad,0dba6b011d
1,96ACBBA7ED-01,Bad,96acbba7ed
2,289421D82A-01,Bad,289421d82a
3,9ECE67D6C5-01,Good,9ece67d6c5
4,5C2C402094-01,Bad,5c2c402094
5,E6254CAD30-01,Good,e6254cad30
6,AE03151FEE-01,Good,ae03151fee
7,E55A9A7293-01,Bad,e55a9a7293
8,CFE9E0DC5E-01,Good,cfe9e0dc5e
9,3744B9845D-01,Good,3744b9845d


Unmatched application records: 7


,customer_id
105,41319800000000
207,2101485999999999891963124957726643363108066977...
467,7752f74e8c
512,805206719d
567,902431691
570,689b7958a2
620,e2ef95a963


Customers with conflicting loan outcomes: 3


,idLoan,flgGood,customer_id_check
163,B1A2837477-01,Good,b1a2837477
511,B1A2837477-02,Bad,b1a2837477
120,B3F9914DBB-01,Good,b3f9914dbb
1043,B3F9914DBB-02,Bad,b3f9914dbb
23,DFB2F1EC63-01,Good,dfb2f1ec63
631,DFB2F1EC63-02,Bad,dfb2f1ec63


Application columns:
['customer_id', 'amount_requested', 'birth_date', 'status', 'residence_rent_or_own', 'monthly_rent_amount', 'bank_account_direct_deposit', 'application_when', 'loan_duration', 'payment_ach', 'num_payments', 'payment_amount', 'amount_approved', 'duration_approved', 'payment_amount_approved', 'address_zip', 'email', 'bank_routing_number', 'email_duration', 'residence_duration', 'bank_account_duration', 'payment_frequency', 'home_phone_type', 'other_phone_type', 'how_use_money', 'monthly_income_amount', 'raw_l2c_score', 'raw_FICO_telecom', 'raw_FICO_retail', 'raw_FICO_bank_card', 'raw_FICO_money']
